<a href="https://colab.research.google.com/github/igorfantucci/Aula-Automatica---GRUPO-5/blob/main/etapa-01-logica/10%20-%20Avaliacao%20Modulo%201%20Motor%20de%20Intertravamento%20e%20Diagnostico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 10 - Notebook: Avaliação Integrada do Módulo 1 — SCADA-Core Segurança & Diagnóstico
## Processo: Planta Industrial de Produção de Biodiesel (Transesterificação em Batelada)
### Grupo 5 — Automação de Processos Químicos

Neste notebook executável consolidamos a arquitetura completa do **SCADA-Core Módulo 1** para a **Planta de Produção de Biodiesel**. Integramos as cinco camadas de supervisão, controle lógico e inteligência baseada em regras:

1. **Telemetria de Sensores ISA-5.1 e Discretização Proposicional:** Mapeamento de sinais analógicos de campo e status de atuadores para variáveis proposicionais booleanas;
2. **Motor de Permissivos e Intertravamentos Failsafe:** Avaliação determinística de condições de autorização e desarmes (*Trips*) de segurança para os equipamentos dos Setores 100, 200, 300 e 400;
3. **Base de Conhecimento Especialista (RBS):** Catálogo formal em Cláusulas de Horn com 8 regras de diagnóstico de causa-raiz (R-01 a R-08), priorização SIL (10 a 7) e Procedimentos Operacionais Padrão (POP);
4. **Motor de Inferência Progressivo (*Forward Chaining*):** Raciocínio multinível e em cascata com resolução determinística de conflitos por severidade;
5. **Suíte de Testes de Estresse Operacional:** Simulação e validação com 100% de cobertura sobre 5 cenários críticos da planta.

In [1]:
def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

import time
from dataclasses import dataclass, field
from typing import Dict, List, Set, Tuple, Any, Optional

@dataclass
class Fato:
    nome: str
    valor: bool
    descricao: str
    fonte: str = "SENSOR"  # 'SENSOR' ou 'INFERIDO'
    timestamp: float = field(default_factory=time.time)

@dataclass
class RegraDiagnostico:
    id_regra: str
    antecedentes: Set[str]
    consequente: str
    descricao_diagnostico: str
    severidade: str       # 'CRÍTICA', 'ALTA', 'MÉDIA', 'BAIXA'
    prioridade: int       # 1 a 10 (10 = mais urgente / SIL 3)
    tempo_resposta_max_s: float
    procedimento_pop: str

class MapeadorProposicionalBiodiesel:
    """
    Converte telemetria analógica e estados discretos de sensores ISA-5.1
    em proposições booleanas da Planta de Produção de Biodiesel.
    """
    def extrair_proposicoes(self, telemetria: Dict[str, float]) -> Dict[str, bool]:
        return {
            # Setor 100: Metanol e Metóxido
            'g_alm': telemetria.get('AT-100', 0.0) >= 20.0,            # Vapores metanol >= 20 ppm
            'l_oleo': telemetria.get('LT-101', 0.0) >= 80.0,          # Óleo suficiente
            'l_met': telemetria.get('LT-102', 0.0) >= 80.0,           # Metanol suficiente
            'l_mix': telemetria.get('LT-103', 0.0) >= 90.0,           # Metóxido pronto
            'm_mix': bool(telemetria.get('AG-103', 0)),               # Misturador metóxido
            'v_met': bool(telemetria.get('XV-102', 0)),               # Válvula metanol
            'b_met': bool(telemetria.get('P-102', 0)),                # Bomba metanol
            
            # Setor 200: Reator R-200
            'p1': telemetria.get('PT-201', 0.0) >= 2.5,               # Sobrepressão reator >= 2.5 bar
            't_alta': telemetria.get('TT-201', 0.0) >= 65.0,          # Temperatura crítica >= 65 °C
            't_proc': telemetria.get('TT-201', 0.0) >= 55.0,          # Temperatura de processo >= 55 °C
            'l_alto': telemetria.get('LT-201', 0.0) >= 95.0,          # Nível alto / transbordamento >= 95%
            'l_baixo': telemetria.get('LT-201', 0.0) <= 15.0,         # Nível baixo <= 15%
            'l_reator': telemetria.get('LT-201', 0.0) >= 80.0,        # Nível normal de batelada >= 80%
            'm_reator': bool(telemetria.get('AG-201', 0)),            # Agitador reator R-200
            'h1': bool(telemetria.get('HT-201', 0)),                  # Aquecimento ligado/autorizado
            'r1': telemetria.get('CW-201', 0.0) >= 3.0,               # Resfriamento emergência OK (>= 3 bar)
            'v_in_oleo': bool(telemetria.get('XV-201', 0)),           # Válvula óleo aberta
            'v_in_mix': bool(telemetria.get('XV-202', 0)),            # Válvula metóxido aberta
            'v_out_r': bool(telemetria.get('XV-203', 0)),             # Válvula saída reator
            
            # Setor 300: Decantação e Separação
            'l_dec': telemetria.get('LT-301', 0.0) >= 90.0,           # Decantador cheio
            'i_glic': bool(telemetria.get('IT-301', 0)),              # Interface glicerina detectada
            'v_glic': bool(telemetria.get('XV-301', 0)),              # Dreno glicerina aberto
            'v_bruto': bool(telemetria.get('XV-302', 0)),             # Saída biodiesel bruto
            
            # Setor 400: Purificação e Expedição
            'f_lav': telemetria.get('FS-401', 0.0) >= 5.0,            # Fluxo água lavagem >= 5 L/min
            'l_fim': telemetria.get('LT-402', 0.0) >= 95.0,           # Tanque final cheio
            'v_final': bool(telemetria.get('XV-401', 0)),             # Entrada tanque final
            'b_final': bool(telemetria.get('P-401', 0)),              # Bomba transferência final
            
            # Parada Geral de Emergência
            'e1': bool(telemetria.get('ESD-100', 0))                  # Botoeira ESD
        }

class BaseConhecimentoBiodiesel:
    def __init__(self):
        self.regras: List[RegraDiagnostico] = []
        self._indice_antecedentes: Dict[str, List[RegraDiagnostico]] = {}
        self._carregar_catalogo_padrao()

    def adicionar_regra(
        self, id_regra: str, antecedentes: List[str], consequente: str,
        descricao: str, severidade: str = "ALTA", prioridade: int = 5,
        tempo_max_s: float = 5.0, pop: str = "Verificar malha"
    ):
        regra = RegraDiagnostico(
            id_regra=id_regra,
            antecedentes=set(antecedentes),
            consequente=consequente,
            descricao_diagnostico=descricao,
            severidade=severidade,
            prioridade=prioridade,
            tempo_resposta_max_s=tempo_max_s,
            procedimento_pop=pop
        )
        self.regras.append(regra)
        for ant in antecedentes:
            if ant not in self._indice_antecedentes:
                self._indice_antecedentes[ant] = []
            self._indice_antecedentes[ant].append(regra)

    def _carregar_catalogo_padrao(self):
        # R-01: Runaway Térmico e Sobrepressão no Reator R-200
        self.adicionar_regra(
            "R-01", ["p1", "t_alta"], "EXOTERMIA_RUNAWAY_REATOR",
            "Exotermia Descontrolada e Sobrepressão no Reator R-200", "CRÍTICA", 10, 0.5,
            "POP-SIS-01: Desarme total de HT-201, corte de XV-202 e abertura plena de CW-201"
        )
        # R-02: Corte Imediato de Metóxido
        self.adicionar_regra(
            "R-02", ["EXOTERMIA_RUNAWAY_REATOR", "v_in_mix"], "TRIP_ALIMENTACAO_METOXIDO",
            "Corte Imediato da Dosagem de Metóxido por Reação Fora de Controle", "CRÍTICA", 10, 0.5,
            "POP-SIS-02: Fechar imediatamente XV-202, desenergizar P-102 e inertizar com N2"
        )
        # R-03: Fuga de Vapores de Metanol Setor 100
        self.adicionar_regra(
            "R-03", ["g_alm"], "VAZAMENTO_GAS_METANOL_S100",
            "Detecção de Vapores Inflamáveis/Tóxicos de Metanol no Setor 100", "CRÍTICA", 9, 1.0,
            "POP-SST-03: Cortar XV-102 e XV-202, ligar exaustão e desenergizar bombas P-102"
        )
        # R-04: Transbordamento Reator por Óleo
        self.adicionar_regra(
            "R-04", ["l_alto", "v_in_oleo"], "TRANSBORDAMENTO_REATOR_R200",
            "Sobrecarga Volumétrica de Óleo Vegetal no Reator de Transesterificação", "CRÍTICA", 9, 1.0,
            "POP-PR-01: Fechar XV-201, desligar bomba de óleo P-101 e reter batelada"
        )
        # R-05: Inibição de Aquecimento sem Resfriamento de Emergência
        self.adicionar_regra(
            "R-05", ["h1", "not_r1"], "OPERACAO_TERMICA_SEM_SALVAGUARDA",
            "Acionamento de Aquecedor HT-201 sem Circuito de Resfriamento de Emergência Disponível", "CRÍTICA", 9, 1.0,
            "POP-SIS-04: Trip imediato de HT-201 e alarme de manutenção na linha CW-201"
        )
        # R-06: Risco de Cavitação no Agitador
        self.adicionar_regra(
            "R-06", ["l_baixo", "m_reator"], "RISCO_CAVITACAO_AGITADOR_R200",
            "Operação do Agitador sem Carga Hidráulica Mínima no Reator R-200", "ALTA", 8, 2.0,
            "POP-MA-05: Desarmar inversor de AG-201 e bloquear aquecimento HT-201"
        )
        # R-07: Perda de Biodiesel no Dreno de Glicerina
        self.adicionar_regra(
            "R-07", ["v_glic", "not_i_glic"], "PERDA_BIODIESEL_DRENO_GLICERINA",
            "Drenagem Indevida de Biodiesel Bruto pela Linha de Fundo de Glicerina", "ALTA", 8, 1.5,
            "POP-SEP-02: Fechar válvula proporcional XV-301 e reajustar tempo de decantação"
        )
        # R-08: Transferência Final sem Fluxo de Lavagem
        self.adicionar_regra(
            "R-08", ["b_final", "not_f_lav"], "IMPUREZA_CATALISADOR_BIODIESEL",
            "Transferência de Biodiesel sem Etapa de Lavagem e Neutralização Concluída", "ALTA", 7, 3.0,
            "POP-PUR-04: Bloquear bomba P-401, fechar XV-401 e restabelecer água desmineralizada"
        )

    def exportar_catalogo(self) -> List[Dict[str, Any]]:
        return [{
            "ID": r.id_regra,
            "Prioridade": r.prioridade,
            "Severidade": r.severidade,
            "SE (Antecedentes)": " AND ".join(sorted(r.antecedentes)),
            "ENTÃO (Consequente)": r.consequente,
            "Diagnóstico": r.descricao_diagnostico,
            "POP": r.procedimento_pop
        } for r in sorted(self.regras, key=lambda x: x.prioridade, reverse=True)]

class MotorInferenciaForward:
    def __init__(self, base_conhecimento: BaseConhecimentoBiodiesel):
        self.bc = base_conhecimento

    def inferir(self, fatos_iniciais: Set[str]) -> Tuple[Set[str], List[Dict[str, Any]], List[str]]:
        fatos_conhecidos = set(fatos_iniciais)
        trilha_deducao = []
        pops_acionados = []
        passo = 1
        novos_fatos = True

        while novos_fatos:
            novos_fatos = False
            regras_candidatas = sorted(self.bc.regras, key=lambda r: r.prioridade, reverse=True)
            for regra in regras_candidatas:
                if regra.antecedentes.issubset(fatos_conhecidos) and regra.consequente not in fatos_conhecidos:
                    fatos_conhecidos.add(regra.consequente)
                    trilha_deducao.append({
                        "Passo": passo,
                        "ID Regra": regra.id_regra,
                        "Prioridade": regra.prioridade,
                        "Severidade": regra.severidade,
                        "Fato Inferido": regra.consequente,
                        "Diagnóstico": regra.descricao_diagnostico
                    })
                    if regra.procedimento_pop not in pops_acionados:
                        pops_acionados.append(regra.procedimento_pop)
                    passo += 1
                    novos_fatos = True
                    break
        return fatos_conhecidos, trilha_deducao, pops_acionados

class SCADACoreModulo1Biodiesel:
    def __init__(self):
        self.mapeador = MapeadorProposicionalBiodiesel()
        self.bc = BaseConhecimentoBiodiesel()
        self.motor = MotorInferenciaForward(self.bc)

    def processar_ciclo_scan(self, telemetria: Dict[str, float]) -> Dict[str, Any]:
        props = self.mapeador.extrair_proposicoes(telemetria)
        
        # 1. Permissivos e Trips dos Atuadores Críticos
        permissivo_ht201 = props['r1'] and props['m_reator'] and props['l_reator'] and not props['t_alta'] and not props['p1'] and not props['e1']
        trip_ht201 = not props['r1'] or not props['m_reator'] or not props['l_reator'] or props['t_alta'] or props['p1'] or props['e1']
        
        permissivo_xv202 = props['m_reator'] and props['l_reator'] and not props['p1'] and not props['t_alta'] and not props['l_alto'] and not props['g_alm'] and not props['e1']
        trip_xv202 = not props['m_reator'] or not props['l_reator'] or props['p1'] or props['t_alta'] or props['l_alto'] or props['g_alm'] or props['e1']
        
        permissivo_ag103 = props['l_mix'] and not props['g_alm'] and not props['e1']
        trip_ag103 = not props['l_mix'] or props['g_alm'] or props['e1']
        
        permissivo_xv301 = props['l_dec'] and props['i_glic'] and not props['e1']
        trip_xv301 = not props['l_dec'] or not props['i_glic'] or props['e1']
        
        permissivo_p401 = props['f_lav'] and props['v_final'] and not props['l_fim'] and not props['e1']
        trip_p401 = not props['f_lav'] or not props['v_final'] or props['l_fim'] or props['e1']
        
        trip_global_esd = props['e1']

        # Fatos booleanos ativos para o motor de inferência
        fatos_ativos = {k for k, v in props.items() if v}
        if not props['r1']:
            fatos_ativos.add('not_r1')
        if not props['i_glic']:
            fatos_ativos.add('not_i_glic')
        if not props['f_lav']:
            fatos_ativos.add('not_f_lav')
            
        fatos_inf, trilha, pops = self.motor.inferir(fatos_ativos)
        diagnosticos_inferidos = [f for f in fatos_inf if f not in fatos_ativos]
        
        return {
            "Telemetria_Entrada": telemetria,
            "Proposicoes_Mapeadas": props,
            "Intertravamentos": {
                "Permissivo_HT-201": permissivo_ht201,
                "Trip_HT-201": trip_ht201,
                "Permissivo_XV-202": permissivo_xv202,
                "Trip_XV-202": trip_xv202,
                "Permissivo_AG-103": permissivo_ag103,
                "Trip_AG-103": trip_ag103,
                "Permissivo_XV-301": permissivo_xv301,
                "Trip_XV-301": trip_xv301,
                "Permissivo_P-401": permissivo_p401,
                "Trip_P-401": trip_p401,
                "Trip_Global_ESD": trip_global_esd
            },
            "Diagnosticos_CausaRaiz": diagnosticos_inferidos,
            "Trilha_Deducao": trilha,
            "POPs_Despachados": pops
        }

# Inicialização e Exibição do Catálogo
core = SCADACoreModulo1Biodiesel()
print("=== CATÁLOGO OFICIAL DA BASE DE CONHECIMENTO DO SCADA-CORE (BIODIESEL) ===")
print(formatar_tabela(core.bc.exportar_catalogo()))
print("\n[OK] Módulo SCADA-Core Biodiesel inicializado com sucesso!")

=== CATÁLOGO OFICIAL DA BASE DE CONHECIMENTO DO SCADA-CORE (BIODIESEL) ===
ID   | Prioridade | Severidade | SE (Antecedentes)                     | ENTÃO (Consequente)              | Diagnóstico                                                                           | POP                                                                                
-----+------------+------------+---------------------------------------+----------------------------------+---------------------------------------------------------------------------------------+------------------------------------------------------------------------------------
R-01 | 10         | CRÍTICA    | p1 AND t_alta                         | EXOTERMIA_RUNAWAY_REATOR         | Exotermia Descontrolada e Sobrepressão no Reator R-200                                | POP-SIS-01: Desarme total de HT-201, corte de XV-202 e abertura plena de CW-201    
R-02 | 10         | CRÍTICA    | EXOTERMIA_RUNAWAY_REATOR AND v_in_mix | TRIP_ALIMENT

## 2. Bateria de Testes de Estresse: Simulação dos 5 Cenários Industriais

Abaixo submetemos o **SCADA-Core Módulo 1** aos 5 cenários críticos operacionais:

1. **Cenário 1 (Operação Nominal):** Batelada regular no reator R-200. Temperatura em $58^\circ\text{C}$, pressão em $1.2\text{ bar}$, resfriamento em $3.8\text{ bar}$, agitador ligado e nível adequado. Permissivo de aquecimento ativo e zero trips;
2. **Cenário 2 (Runaway Térmico e Sobrepressão no Reator):** Exotermia descontrolada ($p_1 = 1, t_{\text{alta}} = 1$) com válvula de metóxido aberta ($v_{\text{in\_mix}} = 1$). O motor *Forward Chaining* ativa em cascata as regras **R-01** e **R-02**, comanda o desarme imediato de $\text{HT-201}$ e $\text{XV-202}$ e despacha os **POP-SIS-01** e **POP-SIS-02**;
3. **Cenário 3 (Vazamento de Vapores de Metanol no Setor 100):** Detector $\text{AT-100}$ registra concentração crítica ($g_{\text{alm}} = 1$). O SCADA ativa a regra **R-03**, bloqueia a dosagem de metanol e despacha o **POP-SST-03**;
4. **Cenário 4 (Cavitação / Partida a Seco do Agitador no Reator):** Agitador $\text{AG-201}$ acionado com nível de líquido em estado crítico ($l_{\text{baixo}} = 1$). Ativação de **R-06**, desarme do inversor e POP-MA-05;
5. **Cenário 5 (Drenagem sem Interface & Parada Global ESD):** Válvula de dreno de fundo $\text{XV-301}$ aberta sem leitura válida do sensor $\text{IT-301}$ e acionamento da botoeira $\text{ESD-100}$. Ativação de **R-07**, desarme de $\text{XV-301}$, trip global failsafe em todos os setores e **POP-SEP-02**.

In [2]:
# Definição da Telemetria dos 5 Cenários de Teste
cenarios = {
    "1. Operação Nominal Estável": {
        'PT-201': 1.2, 'TT-201': 58.0, 'LT-201': 85.0, 'AT-100': 0.0,
        'CW-201': 3.8, 'AG-201': 1.0, 'HT-201': 1.0, 'XV-201': 0.0, 'XV-202': 0.0,
        'LT-103': 92.0, 'AG-103': 1.0, 'LT-301': 92.0, 'IT-301': 1.0, 'XV-301': 1.0,
        'FS-401': 6.5, 'LT-402': 50.0, 'XV-401': 1.0, 'P-401': 1.0, 'ESD-100': 0.0
    },
    "2. Runaway Térmico e Sobrepressão no Reator": {
        'PT-201': 2.8, 'TT-201': 68.5, 'LT-201': 88.0, 'AT-100': 0.0,
        'CW-201': 3.5, 'AG-201': 1.0, 'HT-201': 1.0, 'XV-202': 1.0, 'ESD-100': 0.0
    },
    "3. Vazamento de Metanol Setor 100": {
        'PT-201': 1.2, 'TT-201': 58.0, 'LT-201': 85.0, 'AT-100': 28.0,
        'CW-201': 3.5, 'AG-201': 1.0, 'HT-201': 0.0, 'XV-102': 1.0, 'P-102': 1.0, 'ESD-100': 0.0
    },
    "4. Partida a Seco Agitador Reator": {
        'PT-201': 1.0, 'TT-201': 25.0, 'LT-201': 10.0, 'AT-100': 0.0,
        'CW-201': 3.5, 'AG-201': 1.0, 'HT-201': 0.0, 'ESD-100': 0.0
    },
    "5. Dreno sem Interface e Parada Geral ESD": {
        'PT-201': 1.2, 'TT-201': 55.0, 'LT-201': 80.0, 'AT-100': 0.0,
        'CW-201': 3.5, 'LT-301': 92.0, 'IT-301': 0.0, 'XV-301': 1.0, 'ESD-100': 1.0
    }
}

resultados_resumo = []
res_cenarios_completos = {}

for nome_cenario, telemetria in cenarios.items():
    res = core.processar_ciclo_scan(telemetria)
    res_cenarios_completos[nome_cenario] = res
    diag_str = ", ".join(res["Diagnosticos_CausaRaiz"]) if res["Diagnosticos_CausaRaiz"] else "NENHUM"
    resultados_resumo.append({
        "Cenário": nome_cenario,
        "Permissivo HT-201": res["Intertravamentos"]["Permissivo_HT-201"],
        "Trip HT-201": res["Intertravamentos"]["Trip_HT-201"],
        "Trip XV-202": res["Intertravamentos"]["Trip_XV-202"],
        "Trip Global ESD": res["Intertravamentos"]["Trip_Global_ESD"],
        "Diagnósticos Inferidos": diag_str
    })

print("=== RELATÓRIO DE EXECUÇÃO DOS 5 CENÁRIOS DE ESTRESSE OPERACIONAL (SCADA-CORE) ===")
print(formatar_tabela(resultados_resumo))

print("\n=== AUDITORIA DA TRILHA DE INFERÊNCIA FORWARD CHAINING (CENÁRIO 2) ===")
print(formatar_tabela(res_cenarios_completos["2. Runaway Térmico e Sobrepressão no Reator"]["Trilha_Deducao"]))

# Asserções Formais de Validação do Módulo 1
# Cenário 1: Operação Normal
assert res_cenarios_completos["1. Operação Nominal Estável"]["Intertravamentos"]["Permissivo_HT-201"] is True
assert res_cenarios_completos["1. Operação Nominal Estável"]["Intertravamentos"]["Trip_HT-201"] is False
assert len(res_cenarios_completos["1. Operação Nominal Estável"]["Diagnosticos_CausaRaiz"]) == 0

# Cenário 2: Runaway Térmico
assert res_cenarios_completos["2. Runaway Térmico e Sobrepressão no Reator"]["Intertravamentos"]["Trip_HT-201"] is True
assert res_cenarios_completos["2. Runaway Térmico e Sobrepressão no Reator"]["Intertravamentos"]["Trip_XV-202"] is True
assert "EXOTERMIA_RUNAWAY_REATOR" in res_cenarios_completos["2. Runaway Térmico e Sobrepressão no Reator"]["Diagnosticos_CausaRaiz"]
assert "TRIP_ALIMENTACAO_METOXIDO" in res_cenarios_completos["2. Runaway Térmico e Sobrepressão no Reator"]["Diagnosticos_CausaRaiz"]

# Cenário 3: Vazamento Metanol
assert res_cenarios_completos["3. Vazamento de Metanol Setor 100"]["Intertravamentos"]["Trip_XV-202"] is True
assert "VAZAMENTO_GAS_METANOL_S100" in res_cenarios_completos["3. Vazamento de Metanol Setor 100"]["Diagnosticos_CausaRaiz"]

# Cenário 4: Partida a Seco Agitador
assert "RISCO_CAVITACAO_AGITADOR_R200" in res_cenarios_completos["4. Partida a Seco Agitador Reator"]["Diagnosticos_CausaRaiz"]

# Cenário 5: Dreno sem Interface e ESD
assert res_cenarios_completos["5. Dreno sem Interface e Parada Geral ESD"]["Intertravamentos"]["Trip_Global_ESD"] is True
assert "PERDA_BIODIESEL_DRENO_GLICERINA" in res_cenarios_completos["5. Dreno sem Interface e Parada Geral ESD"]["Diagnosticos_CausaRaiz"]

print("\n[SUCESSO] 100% dos cenários operacionais, intertravamentos e diagnósticos foram validados com exatidão!")

=== RELATÓRIO DE EXECUÇÃO DOS 5 CENÁRIOS DE ESTRESSE OPERACIONAL (SCADA-CORE) ===
Cenário                                     | Permissivo HT-201 | Trip HT-201 | Trip XV-202 | Trip Global ESD | Diagnósticos Inferidos                                          
--------------------------------------------+-------------------+-------------+-------------+-----------------+-----------------------------------------------------------------
1. Operação Nominal Estável                 | True              | False       | False       | False           | NENHUM                                                          
2. Runaway Térmico e Sobrepressão no Reator | False             | True        | True        | False           | EXOTERMIA_RUNAWAY_REATOR, TRIP_ALIMENTACAO_METOXIDO             
3. Vazamento de Metanol Setor 100           | True              | False       | True        | False           | VAZAMENTO_GAS_METANOL_S100                                      
4. Partida a Seco Agitador Reator